# Othello — Training and evaluation of agents

## 1 — Imports

In [9]:
import sys
sys.path.insert(0, ".")          # adjust to your project path

import numpy as np
import matplotlib.pyplot as plt

from train              import train
from train_multi_agent  import train_multi_agent
from evaluation         import evaluate_fair, play_game

from agents.random_agent    import RandomAgent
from agents.greedy_agent    import GreedyAgent
from agents.heuristic_agent import HeuristicAgent
from agents.minimax_agent   import MinimaxAgent
from agent import DQNAgent

OPPONENT_CLASSES = {
    "random":    RandomAgent,
    "greedy":    GreedyAgent,
    "heuristic": HeuristicAgent,
    "minimax":   MinimaxAgent,
}

print("Imports OK")

Imports OK


## 2 — Configuration

In [8]:
BOARD_SIZE    = 6
NUM_EPISODES  = 5_000
MODELS_DIR    = "models"

## 3 — Training agents

Each `train()` call saves the model to disk and returns a dictionary with history.  
Comment out agents you don't want to train.

In [ ]:
# Classic DQN
result_dqn = train(
    board_size    = BOARD_SIZE,
    num_episodes  = NUM_EPISODES,
    heuristic_weight = 0.0,
    use_per       = False,
    opponent_type = "mix",
    model_path    = f"{MODELS_DIR}/dqn_6.pth",
    print_every   = 500,
    eval_every    = 1_000,
    save_every    = 1_000,
)
agent_dqn = result_dqn["agent"]


Training: dqn | hw=0.0 | per=False | opponent=mix | episodes=5000
[   500/5000] opp=heuristic  ε=0.606 | avg_r=0.030 win%=0.500 | loss=0.0072 | W/D/L=227/18/255
[  1000/5000] opp=heuristic  ε=0.368 | avg_r=0.080 win%=0.520 | loss=0.0095 | W/D/L=492/48/460

── Evaluation (no exploration) ──
  random     score=0.715 win=0.680 W/D/L=68/7/25
  greedy     score=0.500 win=0.500 W/D/L=50/0/50
  heuristic  score=1.000 win=1.000 W/D/L=100/0/0
  minimax    score=0.000 win=0.000 W/D/L=0/0/100

[  1500/5000] opp=minimax    ε=0.223 | avg_r=0.220 win%=0.600 | loss=0.0050 | W/D/L=780/65/655
[  2000/5000] opp=minimax    ε=0.135 | avg_r=0.010 win%=0.480 | loss=0.0106 | W/D/L=990/81/929

── Evaluation (no exploration) ──
  random     score=0.705 win=0.670 W/D/L=67/7/26
  greedy     score=0.500 win=0.500 W/D/L=50/0/50
  heuristic  score=0.250 win=0.000 W/D/L=0/50/50
  minimax    score=0.000 win=0.000 W/D/L=0/0/100

[  2500/5000] opp=greedy     ε=0.082 | avg_r=-0.230 win%=0.340 | loss=0.0200 | W/D/L=1206/

In [7]:
# Guided DQN  (heuristic_weight > 0)
result_guided = train(
    board_size    = BOARD_SIZE,
    num_episodes  = NUM_EPISODES,
    heuristic_weight = 0.2,
    use_per       = False,
    opponent_type = "mix",
    model_path    = f"{MODELS_DIR}/guided_dqn.pth",
    print_every   = 500,
    eval_every    = 1_000,
    save_every    = 1_000,
)
agent_guided = result_guided["agent"]


Training: guided_dqn | hw=0.2 | per=False | opponent=mix | episodes=5000
[   500/5000] opp=random     ε=0.606 | avg_r=0.170 win%=0.570 | loss=0.0053 | W/D/L=237/26/237
[  1000/5000] opp=random     ε=0.368 | avg_r=0.400 win%=0.660 | loss=0.0083 | W/D/L=569/44/387

── Evaluation (no exploration) ──
  random     score=0.960 win=0.950 W/D/L=95/2/3
  greedy     score=1.000 win=1.000 W/D/L=100/0/0
  heuristic  score=1.000 win=1.000 W/D/L=100/0/0
  minimax    score=0.150 win=0.150 W/D/L=15/0/85

[  1500/5000] opp=self-play  ε=0.223 | avg_r=0.410 win%=0.680 | loss=0.0089 | W/D/L=922/60/518
[  2000/5000] opp=self-play  ε=0.135 | avg_r=0.150 win%=0.560 | loss=0.0064 | W/D/L=1177/70/753

── Evaluation (no exploration) ──
  random     score=0.935 win=0.930 W/D/L=93/1/6
  greedy     score=1.000 win=1.000 W/D/L=100/0/0
  heuristic  score=0.500 win=0.500 W/D/L=50/0/50
  minimax    score=0.000 win=0.000 W/D/L=0/0/100

[  2500/5000] opp=heuristic  ε=0.082 | avg_r=-0.110 win%=0.440 | loss=0.0107 | W/D/L

In [6]:
# PER DQN  (prioritized replay)
result_per = train(
    board_size    = BOARD_SIZE,
    num_episodes  = NUM_EPISODES,
    heuristic_weight = 0.0,
    use_per       = True,
    opponent_type = "mix",
    #load_model_path = f"{MODELS_DIR}/per_dqn_6.pth",
    model_path    = f"{MODELS_DIR}/per_dqn_6.pth",
    print_every   = 500,
    eval_every    = 1_000,
    save_every    = 1_000,
)
agent_per = result_per["agent"]


Training: per_dqn | hw=0.0 | per=True | opponent=mix | episodes=5000
[   500/5000] opp=greedy     ε=0.606 | avg_r=-0.060 win%=0.440 | loss=0.0050 | beta=0.442 | td=0.1190 | W/D/L=181/26/293
[  1000/5000] opp=random     ε=0.368 | avg_r=0.020 win%=0.470 | loss=0.0047 | beta=0.491 | td=0.1166 | W/D/L=446/51/503

── Evaluation (no exploration) ──
  random     score=0.700 win=0.670 W/D/L=67/6/27
  greedy     score=1.000 win=1.000 W/D/L=100/0/0
  heuristic  score=0.000 win=0.000 W/D/L=0/0/100
  minimax    score=0.000 win=0.000 W/D/L=0/0/100

[  1500/5000] opp=minimax    ε=0.223 | avg_r=0.150 win%=0.550 | loss=0.0022 | beta=0.540 | td=0.1034 | W/D/L=731/73/696
[  2000/5000] opp=greedy     ε=0.135 | avg_r=-0.040 win%=0.480 | loss=0.0038 | beta=0.589 | td=0.1251 | W/D/L=929/91/980

── Evaluation (no exploration) ──
  random     score=0.695 win=0.670 W/D/L=67/5/28
  greedy     score=1.000 win=1.000 W/D/L=100/0/0
  heuristic  score=1.000 win=1.000 W/D/L=100/0/0
  minimax    score=0.075 win=0.050 

In [8]:
# Guided + PER
result_guided_per = train(
    board_size    = BOARD_SIZE,
    num_episodes  = NUM_EPISODES,
    heuristic_weight = 0.2,
    use_per       = True,
    opponent_type = "mix",
    model_path    = f"{MODELS_DIR}/guided_per_dqn_6.pth",
    print_every   = 500,
    eval_every    = 1_000,
    save_every    = 1_000,
)
agent_guided_per = result_guided_per["agent"]


Training: guided_per_dqn | hw=0.2 | per=True | opponent=mix | episodes=5000
[   500/5000] opp=random     ε=0.606 | avg_r=0.130 win%=0.540 | loss=0.0046 | beta=0.443 | td=0.0974 | W/D/L=233/22/245
[  1000/5000] opp=random     ε=0.368 | avg_r=0.560 win%=0.770 | loss=0.0039 | beta=0.491 | td=0.1049 | W/D/L=569/39/392

── Evaluation (no exploration) ──
  random     score=0.910 win=0.900 W/D/L=90/2/8
  greedy     score=1.000 win=1.000 W/D/L=100/0/0
  heuristic  score=0.000 win=0.000 W/D/L=0/0/100
  minimax    score=0.150 win=0.150 W/D/L=15/0/85

[  1500/5000] opp=self-play  ε=0.223 | avg_r=0.430 win%=0.700 | loss=0.0020 | beta=0.540 | td=0.0808 | W/D/L=915/55/530
[  2000/5000] opp=minimax    ε=0.135 | avg_r=-0.080 win%=0.440 | loss=0.0015 | beta=0.588 | td=0.0993 | W/D/L=1135/69/796

── Evaluation (no exploration) ──
  random     score=0.945 win=0.940 W/D/L=94/1/5
  greedy     score=1.000 win=1.000 W/D/L=100/0/0
  heuristic  score=1.000 win=1.000 W/D/L=100/0/0
  minimax    score=0.230 win=0

## 3.1 — Training agents 02

Each `train_max_self-play()` call saves the model to disk and returns a dictionary with history.  
Comment out agents you don't want to train.

In [9]:
from train_max_selfplay import train


In [10]:
# Classic DQN
result_dqn = train(
    board_size    = BOARD_SIZE,
    num_episodes  = NUM_EPISODES,
    heuristic_weight = 0.0,
    use_per       = False,
    opponent_type = "mix",
    model_path    = f"{MODELS_DIR}/dqn_6_02.pth",
    load_model_path = f"{MODELS_DIR}/dqn_6.pth",
    print_every   = 500,
    eval_every    = 1_000,
    save_every    = 1_000,
)
agent_dqn = result_dqn["agent"]


Loaded weights from: models/dqn_6.pth
Training: dqn | hw=0.0 | per=False | opponent=mix | episodes=5000
[   500/5000] opp=minimax    ε=0.606 | avg_r=-0.590 win%=0.190 | loss=0.0084 | W/D/L=85/24/391
[  1000/5000] opp=self-play  ε=0.368 | avg_r=-0.400 win%=0.290 | loss=0.0163 | W/D/L=219/47/734

── Evaluation (no exploration) ──
  random     score=0.800 win=0.790 W/D/L=79/2/19
  greedy     score=0.750 win=0.500 W/D/L=50/50/0
  heuristic  score=0.000 win=0.000 W/D/L=0/0/100
  minimax    score=0.030 win=0.030 W/D/L=3/0/97

[  1500/5000] opp=self-play  ε=0.223 | avg_r=-0.010 win%=0.480 | loss=0.0382 | W/D/L=390/69/1041
[  2000/5000] opp=self-play  ε=0.135 | avg_r=-0.170 win%=0.390 | loss=0.0335 | W/D/L=617/87/1296

── Evaluation (no exploration) ──
  random     score=0.820 win=0.800 W/D/L=80/4/16
  greedy     score=1.000 win=1.000 W/D/L=100/0/0
  heuristic  score=0.000 win=0.000 W/D/L=0/0/100
  minimax    score=0.055 win=0.040 W/D/L=4/3/93

[  2500/5000] opp=minimax    ε=0.082 | avg_r=-0.0

In [11]:
# Guided DQN  (heuristic_weight > 0)
result_guided = train(
    board_size    = BOARD_SIZE,
    num_episodes  = NUM_EPISODES,
    heuristic_weight = 0.2,
    use_per       = False,
    opponent_type = "mix",
    load_model_path = f"{MODELS_DIR}/guided_dqn_6.pth",
    model_path    = f"{MODELS_DIR}/guided_dqn_6_02.pth",
    print_every   = 500,
    eval_every    = 1_000,
    save_every    = 1_000,
)
agent_guided = result_guided["agent"]


Loaded weights from: models/guided_dqn_6.pth
Training: guided_dqn | hw=0.2 | per=False | opponent=mix | episodes=5000
[   500/5000] opp=heuristic  ε=0.606 | avg_r=-0.790 win%=0.090 | loss=0.0045 | W/D/L=56/11/433
[  1000/5000] opp=random     ε=0.368 | avg_r=-0.310 win%=0.340 | loss=0.0603 | W/D/L=164/28/808

── Evaluation (no exploration) ──
  random     score=0.925 win=0.900 W/D/L=90/5/5
  greedy     score=0.500 win=0.500 W/D/L=50/0/50
  heuristic  score=0.750 win=0.500 W/D/L=50/50/0
  minimax    score=0.400 win=0.380 W/D/L=38/4/58

[  1500/5000] opp=self-play  ε=0.223 | avg_r=-0.230 win%=0.380 | loss=0.0260 | W/D/L=322/38/1140
[  2000/5000] opp=heuristic  ε=0.135 | avg_r=-0.020 win%=0.480 | loss=0.0218 | W/D/L=515/53/1432

── Evaluation (no exploration) ──
  random     score=0.915 win=0.910 W/D/L=91/1/8
  greedy     score=1.000 win=1.000 W/D/L=100/0/0
  heuristic  score=0.500 win=0.500 W/D/L=50/0/50
  minimax    score=0.090 win=0.090 W/D/L=9/0/91

[  2500/5000] opp=self-play  ε=0.082

In [12]:
# PER DQN  (prioritized replay)
result_per = train(
    board_size    = BOARD_SIZE,
    num_episodes  = NUM_EPISODES,
    heuristic_weight = 0.0,
    use_per       = True,
    opponent_type = "mix",
    load_model_path = f"{MODELS_DIR}/per_dqn_6.pth",
    model_path    = f"{MODELS_DIR}/per_dqn_6_02.pth",
    print_every   = 500,
    eval_every    = 1_000,
    save_every    = 1_000,
)
agent_per = result_per["agent"]


Loaded weights from: models/per_dqn_6.pth
Training: per_dqn | hw=0.0 | per=True | opponent=mix | episodes=5000
[   500/5000] opp=self-play  ε=0.606 | avg_r=-0.490 win%=0.240 | loss=0.0009 | beta=0.924 | td=0.1102 | W/D/L=89/21/390
[  1000/5000] opp=heuristic  ε=0.368 | avg_r=-0.310 win%=0.320 | loss=0.0010 | beta=0.972 | td=0.1751 | W/D/L=245/48/707

── Evaluation (no exploration) ──
  random     score=0.815 win=0.810 W/D/L=81/1/18
  greedy     score=1.000 win=1.000 W/D/L=100/0/0
  heuristic  score=0.500 win=0.500 W/D/L=50/0/50
  minimax    score=0.080 win=0.060 W/D/L=6/4/90

[  1500/5000] opp=heuristic  ε=0.223 | avg_r=-0.110 win%=0.420 | loss=0.0061 | beta=1.000 | td=0.2011 | W/D/L=410/74/1016
[  2000/5000] opp=self-play  ε=0.135 | avg_r=-0.150 win%=0.410 | loss=0.0043 | beta=1.000 | td=0.1696 | W/D/L=605/97/1298

── Evaluation (no exploration) ──
  random     score=0.815 win=0.800 W/D/L=80/3/17
  greedy     score=0.500 win=0.500 W/D/L=50/0/50
  heuristic  score=0.500 win=0.500 W/D/L

In [4]:
# Guided + PER
result_guided_per = train(
    board_size    = BOARD_SIZE,
    num_episodes  = NUM_EPISODES,
    heuristic_weight = 0.2,
    use_per       = True,
    opponent_type = "mix",
    load_model_path = f"{MODELS_DIR}/guided_per_dqn_6.pth",
    model_path    = f"{MODELS_DIR}/guided_per_dqn_6_02.pth",
    print_every   = 500,
    eval_every    = 1_000,
    save_every    = 1_000,
)
agent_guided_per = result_guided_per["agent"]


Loaded weights from: models/guided_per_dqn_6.pth
Training: guided_per_dqn | hw=0.2 | per=True | opponent=mix | episodes=5000
[   500/5000] opp=random     ε=0.606 | avg_r=0.130 win%=0.550 | loss=0.0020 | beta=0.923 | td=0.1258 | W/D/L=248/19/233
[  1000/5000] opp=greedy     ε=0.368 | avg_r=0.410 win%=0.680 | loss=0.0035 | beta=0.972 | td=0.1552 | W/D/L=573/37/390

── Evaluation (no exploration) ──
  random     score=0.995 win=0.990 W/D/L=99/1/0
  greedy     score=1.000 win=1.000 W/D/L=100/0/0
  heuristic  score=0.000 win=0.000 W/D/L=0/0/100
  minimax    score=0.000 win=0.000 W/D/L=0/0/100

[  1500/5000] opp=random     ε=0.223 | avg_r=0.480 win%=0.720 | loss=0.0020 | beta=1.000 | td=0.1273 | W/D/L=924/54/522
[  2000/5000] opp=self-play  ε=0.135 | avg_r=0.170 win%=0.580 | loss=0.0020 | beta=1.000 | td=0.1227 | W/D/L=1190/62/748

── Evaluation (no exploration) ──
  random     score=0.980 win=0.980 W/D/L=98/0/2
  greedy     score=1.000 win=1.000 W/D/L=100/0/0
  heuristic  score=0.750 win=0.

## 4 — Training curves

In [5]:
def moving_avg(arr, w=100):
    return np.convolve(arr, np.ones(w)/w, mode="valid") if len(arr) >= w else np.array(arr)

results = {
    "DQN":        result_dqn,
    "Guided DQN": result_guided,
    "PER DQN":    result_per,
    "Guided PER": result_guided_per,
}

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for name, res in results.items():
    axes[0].plot(moving_avg(res["wins"],  100), label=name)
    axes[1].plot(moving_avg(res["losses"], 50), label=name)

axes[0].set_title("Win rate MA-100")
axes[0].set_xlabel("episode")
axes[0].axhline(0.5, color="gray", linestyle="--", linewidth=0.8)
axes[0].legend()

axes[1].set_title("Loss MA-50")
axes[1].set_xlabel("train step")
axes[1].legend()

plt.tight_layout()
plt.show()

NameError: name 'result_dqn' is not defined

In [11]:
dqn_latest_model_path = f"{MODELS_DIR}/dqn_6_02.pth"
agent_dqn = DQNAgent(
    board_size=BOARD_SIZE,
    heuristic_weight=0.0,
    use_per=False
)
agent_dqn.load(str(dqn_latest_model_path), load_optimizer=False)


guided_latest_model_path = f"{MODELS_DIR}/dqn_6_02.pth"
agent_guided = DQNAgent(
    board_size=BOARD_SIZE,
    heuristic_weight=0.2,
    use_per=False
)
agent_guided.load(str(guided_latest_model_path), load_optimizer=False)

per_latest_model_path = f"{MODELS_DIR}/per_dqn_6_02.pth"
agent_per = DQNAgent(
    board_size=BOARD_SIZE,
    heuristic_weight=0.0,
    use_per=True
)
agent_per.load(str(per_latest_model_path), load_optimizer=False)

guided_per_latest_model_path = f"{MODELS_DIR}/guided_per_dqn_6_02.pth"
agent_guided_per = DQNAgent(
    board_size=BOARD_SIZE,
    heuristic_weight=0.2,
    use_per=True
)
agent_guided_per.load(str(guided_per_latest_model_path), load_optimizer=False)


## 5 — Evaluation vs. rule-based agents

In [12]:
agents = {
    "DQN":        agent_dqn,
    "Guided DQN": agent_guided,
    "PER DQN":    agent_per,
    "Guided PER": agent_guided_per,
}

print(f"{'Agent':<16}", end="")
for opp in OPPONENT_CLASSES:
    print(f"  {opp:>10}", end="")
print(f"  {'average':>8}")
print("-" * 65)

eval_scores = {}
for agent_name, agent in agents.items():
    agent.q_net.eval()
    row = {}
    for opp_name, opp_cls in OPPONENT_CLASSES.items():
        res = evaluate_fair(agent_a=agent, agent_b_class=opp_cls,
                            board_size=BOARD_SIZE, n_games=100)
        row[opp_name] = res["score"]
    eval_scores[agent_name] = row
    agent.q_net.train()

    scores = list(row.values())
    print(f"{agent_name:<16}", end="")
    for s in scores:
        print(f"  {s:>10.3f}", end="")
    print(f"  {np.mean(scores):>8.3f}")

Agent                 random      greedy   heuristic     minimax   average
-----------------------------------------------------------------
DQN                    0.820       0.500       0.000       0.400     0.430
Guided DQN             0.935       1.000       0.500       0.060     0.624
PER DQN                0.845       1.000       0.000       0.070     0.479
Guided PER             0.965       1.000       1.000       0.510     0.869


## 6 — Tournament between DQN agents (round-robin)

In [13]:
from environment import OthelloEnv

N_GAMES = 200   # games per pair

names = list(agents.keys())
n     = len(names)
matrix = np.zeros((n, n))

for i, name_a in enumerate(names):
    for j, name_b in enumerate(names):
        if i == j:
            matrix[i, j] = 0.5
            continue

        ag_a = agents[name_a]
        ag_b = agents[name_b]
        ag_a.q_net.eval()
        ag_b.q_net.eval()

        a_wins = draws = b_wins = 0
        for game_idx in range(N_GAMES):
            role_a = 1 if game_idx % 2 == 0 else -1
            env = OthelloEnv(board_size=BOARD_SIZE)
            obs = env.reset()
            done = False
            while not done:
                if env.current_player == role_a:
                    action = ag_a.select_action(obs)
                else:
                    action = ag_b.select_action(obs)
                obs, _, done, info = env.step(action)
            w = info["winner"]
            if   w == role_a:  a_wins += 1
            elif w == -role_a: b_wins += 1
            else:              draws  += 1

        matrix[i, j] = (a_wins + 0.5 * draws) / N_GAMES

print(f"{'':16}", end="")
for name in names:
    print(f"  {name:>14}", end="")
print()
print("-" * (16 + 16 * n))

for i, name in enumerate(names):
    print(f"{name:<16}", end="")
    for j in range(n):
        print(f"  {matrix[i,j]:>14.3f}", end="")
    print()

                             DQN      Guided DQN         PER DQN      Guided PER
--------------------------------------------------------------------------------
DQN                        0.500           0.000           0.250           0.000
Guided DQN                 1.000           0.500           0.250           0.250
PER DQN                    0.750           0.750           0.500           0.000
Guided PER                 1.000           0.750           1.000           0.500


In [14]:
# Ranking by average score
avg = matrix.mean(axis=1)
ranking = np.argsort(avg)[::-1]

print("\nFinal ranking:")
for rank, idx in enumerate(ranking, 1):
    print(f"  {rank}. {names[idx]:<16}  average score = {avg[idx]:.3f}")


Final ranking:
  1. Guided PER        average score = 0.812
  2. PER DQN           average score = 0.500
  3. Guided DQN        average score = 0.500
  4. DQN               average score = 0.188


## 7 — Multi-agent training (optional)

Alternative to section 3 — all agents train simultaneously,
each episode a learner and opponent are randomly selected.

In [16]:
from train_multi_agent import train_multi_agent
import torch

trained_agents, stats = train_multi_agent(
    board_size    = BOARD_SIZE,
    num_episodes  = NUM_EPISODES,
    model_dir     = f"{MODELS_DIR}/multi",
    print_every   = 500,
)

print("\nStatistics after multi-agent training:")
for name, s in stats.items():
    print(f"  {name:<14}  W/D/L = {s['wins']}/{s['draws']}/{s['losses']}")

[   500/5000] learner=dqn          opp=random     ε=0.368 | reward=0.0 | avg_r_100=-0.670
[  1000/5000] learner=guided_per   opp=dqn        ε=0.135 | reward=1.0 | avg_r_100=0.060
[  1500/5000] learner=guided_per   opp=minimax    ε=0.050 | reward=-1.0 | avg_r_100=0.120
[  2000/5000] learner=guided_per   opp=heuristic  ε=0.050 | reward=1.0 | avg_r_100=0.220
[  2500/5000] learner=guided_per   opp=greedy     ε=0.050 | reward=-1.0 | avg_r_100=0.240
[  3000/5000] learner=per          opp=dqn        ε=0.050 | reward=-1.0 | avg_r_100=-0.070
[  3500/5000] learner=per          opp=greedy     ε=0.050 | reward=0.0 | avg_r_100=0.040
[  4000/5000] learner=dqn          opp=greedy     ε=0.050 | reward=1.0 | avg_r_100=0.050
[  4500/5000] learner=guided_per   opp=per        ε=0.050 | reward=1.0 | avg_r_100=0.280
[  5000/5000] learner=guided       opp=minimax    ε=0.050 | reward=-1.0 | avg_r_100=0.280
Saved dqn → models/multi\dqn.pth
Saved guided → models/multi\guided.pth
Saved per → models/multi\per.pth

AttributeError: 'str' object has no attribute 'items'

## 8 — Loading saved model and StudentAgent

In [18]:
from agent         import DQNAgent
from student_agents.student_agent_minimax_hybrid import StudentAgent

# Load via DQNAgent
loaded = DQNAgent(board_size=BOARD_SIZE)
loaded.load(f"{MODELS_DIR}/guided_per_dqn_6.pth")
loaded.q_net.eval()
print("DQNAgent loaded")

# Load via StudentAgent (submission format)
student = StudentAgent(
    board_size      = BOARD_SIZE,
    checkpoint_path = f"{MODELS_DIR}/guided_per_dqn_6.pth",
)
print("StudentAgent loaded")

# Verification — StudentAgent must always return a legal action
from environment import OthelloEnv
env = OthelloEnv(board_size=BOARD_SIZE)
obs = env.reset()
action = student.select_action(obs)
assert action in obs["legal_actions"]
print(f"select_action → {action}  (legal: {obs['legal_actions']})  ✓")

DQNAgent loaded
StudentAgent loaded
select_action → 9  (legal: [9, 16, 19, 26])  ✓
